# Add E-OBS fields to Extended IBTrACS

In [1]:
# Setup environment
import huracanpy
import xarray as xr
import numpy as np

from tqdm import tqdm

/Users/bourdin/Softs/miniforge3/envs/huracanpy/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
# Script parameters
## Extended IBTrACS file you want to add the attributes to (may be the minimal one or one you already added info to)
IN_FILE = "../files/extended_ibtracs_TS_1980-2022_reaching_BIWE.nc"
## E-OBS file to add
EOBS_FILE = "../../data/E-OBS/wind_percentiles.nc"
old_var_name, new_var_name = "fg", "wind_percentile"
## Output file automatically determined. You may change if you like.
OUT_FILE = IN_FILE[:-3]+"_with_EOBS-"+new_var_name+".nc"

In [3]:
# Open extended IBTrACS
eib = xr.open_dataset(IN_FILE)
#eib["record"] = eib.record # Explicit record as a coordinate

In [4]:
# Open E-OBS file
EOBS_field = xr.open_dataset(EOBS_FILE)[old_var_name].rename(new_var_name)
EOBS_field = EOBS_field.sel(longitude=slice(-11.5,30), latitude = slice(35, 70))

In [5]:
# Select cyclone times from EOBS field
T = eib.time
EOBS_field_sample = EOBS_field.sel(time = T, method = "nearest") 
# NB: EOBS is daily whereas tracks are 3/6-hourly There will be repetitions of daily field.

In [7]:
# Merge the track and sst data
M = xr.merge([eib, EOBS_field_sample.drop_vars("time")],)

In [9]:
M.to_netcdf(OUT_FILE)